# Эксперименты с моделью оценки стоимости квартир

Ноутбук предназначен для ознакомительного анализа датасета. Основной воспроизводимый эксперимент запускается через DVC: `dvc repro`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "part2_dvc").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "part2_dvc").is_dir():
    raise FileNotFoundError("Не найден каталог part2_dvc в репозитории")
DATA_PATH = PROJECT_ROOT / "part2_dvc" / "data" / "raw" / "flats_cleaned.csv"
df = pd.read_csv(DATA_PATH)
df.head()


,flat_id,building_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,price_per_sqm
0,0,6220,1965,6,55.717113,37.781120,2.64,84,12,True,9,9.9,19.900000,1,False,False,35.099998,9500000,270655.282421
1,1,18012,2001,2,55.794849,37.608013,3.00,97,10,True,7,0.0,16.600000,1,False,False,43.000000,13500000,313953.488372
2,2,17821,2000,4,55.740040,37.761742,2.70,80,10,True,9,9.0,32.000000,2,False,False,56.000000,13500000,241071.428571
3,3,18579,2002,4,55.672016,37.570877,2.64,771,17,True,1,10.1,43.099998,3,False,False,76.000000,20000000,263157.894737
4,4,9293,1971,1,55.808807,37.707306,2.60,208,9,True,3,3.0,14.000000,1,False,False,24.000000,5200000,216666.666667


In [2]:
print("Размер:", df.shape)
print("Дубликаты:", df.duplicated().sum())
print("Пропуски:\n", df.isna().sum().sort_values(ascending=False).head(20))


Размер: (138534, 19)
Дубликаты: 0
Пропуски:
 flat_id              0
building_id          0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
price_per_sqm        0
dtype: int64


In [3]:
df["price"].describe(percentiles=[.01, .25, .5, .75, .99])


count    1.385340e+05
mean     1.678502e+07
std      1.909340e+07
min      1.759823e+06
1%       5.450000e+06
25%      8.980000e+06
50%      1.185000e+07
75%      1.680000e+07
99%      1.000000e+08
max      6.020000e+08
Name: price, dtype: float64

In [4]:
df["total_area"].describe(percentiles=[.01, .25, .5, .75, .99])


count    138534.000000
mean         60.957964
std          36.383272
min          11.000000
1%           29.000000
25%          39.099998
50%          52.400002
75%          70.599998
99%         204.868002
max         960.299988
Name: total_area, dtype: float64

## Признаки и целевая переменная

Целевая переменная — `price`. Идентификаторы (`flat_id`, `building_id`) и производный показатель `price_per_sqm` исключаются из признаков, чтобы избежать утечки целевой переменной. Категориальные признаки приводятся к строкам, а пропуски в них обозначаются `__missing__`; CatBoost обрабатывает их как категориальные. Общая подготовка признаков для обучения и оценки находится в `scripts/common.py`.


In [5]:
target = "price"
excluded = {"price", "flat_id", "id", "building_id", "price_per_sqm"}
features = [c for c in df.columns if c not in excluded]
print("Признаки:", features)
print("Целевая переменная:", target)


Признаки: ['build_year', 'building_type_int', 'latitude', 'longitude', 'ceiling_height', 'flats_count', 'floors_total', 'has_elevator', 'floor', 'kitchen_area', 'living_area', 'rooms', 'is_apartment', 'studio', 'total_area']
Целевая переменная: price


## Обучение и оценка

Запускайте из каталога `part2_dvc`:

```bash
dvc repro
dvc metrics show
```

Метрики MAE, RMSE, MAPE и R² появятся в `cv_results/metrics.json`.
